In [ ]:
from collections import defaultdict

# Step 1: Read Corpus
def read_corpus():
    print("Enter tagged corpus (word/TAG). Type 'END' to stop:")
    corpus = []
    while True:
        line = input()
        if line == "END":
            break
        corpus.append(line.strip())
    return corpus


# Step 2: Train HMM
def train_hmm(corpus):
    transition = defaultdict(lambda: defaultdict(int))
    emission = defaultdict(lambda: defaultdict(int))

    START = "<START>"

    for line in corpus:
        tokens = line.split()
        prev_tag = START

        for token in tokens:
            word, tag = token.rsplit("/", 1)

            emission[tag][word] += 1
            transition[prev_tag][tag] += 1

            prev_tag = tag

    # Convert to probabilities
    transition_prob = defaultdict(dict)
    emission_prob = defaultdict(dict)

    for prev in transition:
        total = sum(transition[prev].values())
        for curr in transition[prev]:
            transition_prob[prev][curr] = transition[prev][curr] / total

    for tag in emission:
        total = sum(emission[tag].values())
        for word in emission[tag]:
            emission_prob[tag][word] = emission[tag][word] / total

    return transition_prob, emission_prob


# Step 3: Viterbi Algorithm
def viterbi(words, transition_prob, emission_prob):
    tags = list(emission_prob.keys())
    START = "<START>"

    V = [{}]
    path = {}

    # Initialization
    for tag in tags:
        trans = transition_prob.get(START, {}).get(tag, 0)
        emis = emission_prob.get(tag, {}).get(words[0], 0)

        V[0][tag] = trans * emis
        path[tag] = [tag]

    # Recursion
    for t in range(1, len(words)):
        V.append({})
        new_path = {}

        for curr_tag in tags:
            max_prob = -1
            best_prev = None

            emis = emission_prob.get(curr_tag, {}).get(words[t], 0)

            for prev_tag in tags:
                trans = transition_prob.get(prev_tag, {}).get(curr_tag, 0)
                prob = V[t-1][prev_tag] * trans * emis

                if prob > max_prob:
                    max_prob = prob
                    best_prev = prev_tag

            V[t][curr_tag] = max_prob
            new_path[curr_tag] = path[best_prev] + [curr_tag]

        path = new_path

    # Termination
    max_prob = -1
    best_tag = None

    for tag in tags:
        if V[-1][tag] > max_prob:
            max_prob = V[-1][tag]
            best_tag = tag

    return path[best_tag], max_prob


# ================= MAIN =================

corpus = read_corpus()
transition_prob, emission_prob = train_hmm(corpus)

sentence = input("\nEnter sentence: ")
words = sentence.split()

best_seq, prob = viterbi(words, transition_prob, emission_prob)

print("\nBest Tag Sequence (Viterbi):")
for w, t in zip(words, best_seq):
    print(f"{w} -> {t}")

print(f"\nMax Probability = {prob}")

Enter tagged corpus (word/TAG). Type 'END' to stop:
tall/Adjective boy/Noun eats/Verb quickly/Adverb
girl/Noun sings/Verb sweetly/Adverb
young/Adjective girl/Noun dances/Verb
boy/Noun sings/Verb loudly/Adverb
smart/Adjective student/Noun studies/Verb hard/Adverb
student/Noun writes/Verb notes/Noun
END

Enter sentence: girl sings sweetly

Best Tag Sequence (Viterbi):
girl -> Noun
sings -> Verb
sweetly -> Adverb

Max Probability = 0.009523809523809525
